

هذا النوت بوك يدرّب **AraBERT** على تصنيف سؤال المستخدم إلى:
PLACE, PERSON, EVENT, HERITAGE, OTHER.


In [ ]:
!pip -q install -U transformers datasets accelerate scikit-learn pandas arabert

In [ ]:
import os, json, random, shutil
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)
from arabert.preprocess import ArabertPreprocessor

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "aubmindlab/bert-base-arabertv02"
LABELS = ["PLACE", "PERSON", "EVENT", "HERITAGE", "OTHER"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Model:", MODEL_NAME)

GPU: Tesla T4
Model: aubmindlab/bert-base-arabertv02


In [ ]:
DATA_PATH = "/content/heritage_103_verified.json"

print("Using:", DATA_PATH)

Using: /content/heritage_103_verified.json


In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

if isinstance(raw, dict) and "items" in raw:
    items = raw["items"]
elif isinstance(raw, list):
    items = raw
else:
    raise ValueError("الملف يجب أن يكون list أو dict يحتوي على items.")

print("عدد العناصر:", len(items))
print(json.dumps(items[0], ensure_ascii=False, indent=2)[:1500])

عدد العناصر: 103
{
  "id": "at_turaif",
  "name": "حي الطريف في الدرعية",
  "type": "world_heritage_site",
  "region": "الدرعية - منطقة الرياض",
  "aliases": [
    "حي الطريف",
    "الطريف",
    "At-Turaif",
    "At-Turaif District"
  ],
  "tags": [
    "حي الطريف في الدرعية",
    "الرياض",
    "اليونسكو",
    "تراث عالمي"
  ],
  "short_description": "حي تاريخي في قلب الدرعية، كان مركزًا للحكم في الدولة السعودية الأولى، وأدرجته اليونسكو على قائمة التراث العالمي عام 2010م.",
  "facts": [
    "يقع حي الطريف في قلب مدينة الدرعية بمنطقة الرياض.",
    "ارتبط الحي بالدولة السعودية الأولى ومركز الحكم فيها.",
    "يضم قصر سلوى الذي كان مقرًا للحكم في الدولة السعودية الأولى.",
    "يضم جامع الطريف وعددًا من القصور والمباني التاريخية.",
    "أدرج حي الطريف على قائمة التراث العالمي لليونسكو عام 2010م."
  ],
  "related": [
    "diriyah",
    "salwa_palace",
    "first_saudi_state"
  ],
  "retrieval_terms": [
    "حي الطريف في الدرعية",
    "حي الطريف",
    "الطريف",
    "At-Turaif",
    "At-Turaif

### تحويل أنواع عناصر راوية إلى فئات التصنيف

In [ ]:
TYPE_MAP = {
    "historical_place":"PLACE", "place":"PLACE", "site":"PLACE",
    "archaeological_site":"PLACE", "palace":"PLACE", "fort":"PLACE",
    "village":"PLACE", "district":"PLACE", "landmark":"PLACE",

    "person":"PERSON", "historical_person":"PERSON", "figure":"PERSON",

    "event":"EVENT", "historical_event":"EVENT", "occasion":"EVENT",

    "heritage":"HERITAGE", "intangible_heritage":"HERITAGE",
    "traditional_craft":"HERITAGE", "craft":"HERITAGE",
    "tradition":"HERITAGE", "custom":"HERITAGE", "term":"HERITAGE",
}

def map_type_to_label(t):
    t = str(t or "").strip().lower()
    if t in TYPE_MAP:
        return TYPE_MAP[t]
    if any(x in t for x in ["place","site","palace","fort","village","district"]):
        return "PLACE"
    if any(x in t for x in ["person","figure","imam","king"]):
        return "PERSON"
    if any(x in t for x in ["event","battle","day","state","unification"]):
        return "EVENT"
    if any(x in t for x in ["heritage","craft","tradition","custom","term"]):
        return "HERITAGE"
    return "OTHER"

rows = []
for i, item in enumerate(items):
    name = str(item.get("name","")).strip()
    if not name:
        continue
    aliases = item.get("aliases", [])
    if not isinstance(aliases, list):
        aliases = []
    rows.append({
        "entity_id": item.get("id", f"item_{i}"),
        "name": name,
        "aliases": [str(a).strip() for a in aliases if str(a).strip()],
        "label": map_type_to_label(item.get("type"))
    })

df_entities = pd.DataFrame(rows)
display(df_entities.head())
display(df_entities["label"].value_counts())


EVENT_NAMES = [
    "يوم التأسيس السعودي",
    "اليوم الوطني السعودي"
]

df_entities.loc[
    df_entities["name"].isin(EVENT_NAMES),
    "label"
] = "EVENT"

print(df_entities["label"].value_counts())

,entity_id,name,aliases,label
0,at_turaif,حي الطريف في الدرعية,"[حي الطريف, الطريف, At-Turaif, At-Turaif Distr...",PLACE
1,ar_1a0ce33b86bb,الحِجر (مدائن صالح),[],PLACE
2,ar_c1313b18b0b0,جدة التاريخية، بوابة مكة,[],PLACE
3,ar_2559b40e3039,الفن الصخري في منطقة حائل,[],PLACE
4,ar_4285f4186426,واحة الأحساء,[],PLACE


label
PLACE       71
HERITAGE    22
PERSON       5
EVENT        5
Name: count, dtype: int64

label
PLACE       71
HERITAGE    22
PERSON       5
EVENT        5
Name: count, dtype: int64


##  تقسيم العناصر قبل توليد الأسئلة


In [ ]:
def stratified_entity_split(df, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    train_parts = []
    val_parts = []
    test_parts = []

    for label, group in df.groupby("label"):
        group = group.sample(frac=1, random_state=seed).reset_index(drop=True)

        n = len(group)

        if n < 3:
            train_parts.append(group)
            continue

        n_test = max(1, round(n * test_ratio))
        n_val = max(1, round(n * val_ratio))
        n_train = n - n_val - n_test

        train_parts.append(group.iloc[:n_train])
        val_parts.append(group.iloc[n_train:n_train+n_val])
        test_parts.append(group.iloc[n_train+n_val:])

    train_df = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    test_df = pd.concat(test_parts).sample(frac=1, random_state=seed).reset_index(drop=True)

    return train_df, val_df, test_df


train_entities, val_entities, test_entities = stratified_entity_split(df_entities)

print("TRAIN")
print(train_entities["label"].value_counts())

print("\nVALIDATION")
print(val_entities["label"].value_counts())

print("\nTEST")
print(test_entities["label"].value_counts())

TRAIN
label
PLACE       57
HERITAGE    18
EVENT        3
PERSON       3
Name: count, dtype: int64

VALIDATION
label
PLACE       7
HERITAGE    2
EVENT       1
PERSON      1
Name: count, dtype: int64

TEST
label
PLACE       7
HERITAGE    2
EVENT       1
PERSON      1
Name: count, dtype: int64


In [ ]:
for label in ["PERSON", "EVENT", "OTHER"]:
    print("\n====================")
    print(label)
    print("====================")

    display(
        df_entities[df_entities["label"] == label][
            ["name", "label"]
        ]
    )


PERSON


,name,label
93,الإمام محمد بن سعود,PERSON
96,الملك عبدالعزيز بن عبدالرحمن آل سعود,PERSON
100,الإمام عبدالعزيز بن محمد بن سعود,PERSON
101,الإمام سعود بن عبدالعزيز بن محمد بن سعود,PERSON
102,الإمام تركي بن عبدالله بن محمد بن سعود,PERSON



EVENT


,name,label
94,الدولة السعودية الأولى,EVENT
95,يوم التأسيس السعودي,EVENT
97,معركة استعادة الرياض,EVENT
98,توحيد المملكة العربية السعودية,EVENT
99,اليوم الوطني السعودي,EVENT



OTHER


,name,label


##  توليد أسئلة  متنوعة

In [ ]:
TEMPLATES = {
    "PLACE": [
        "وش قصة {name}؟",
        "احكي لي عن {name}",
        "أين يقع {name}؟",
        "وش تعرف عن {name}؟",
        "أبغى معلومات عن {name}",
        "ما هو {name}؟",
        "حدثيني عن {name}",
        "وش أهمية {name}؟",
        "ودي أعرف وش السالفة ورا {name}",
        "كلمني عن {name}"
    ],

    "PERSON": [
        "من هو {name}؟",
        "من تكون {name}؟",
        "احكي لي عن {name}",
        "وش تعرف عن {name}؟",
        "أبغى معلومات عن {name}",
        "حدثيني عن {name}",
        "عرفيني على {name}",
        "مين {name}؟",
        "ودي أعرف عن {name}"
    ],

    "EVENT": [
        "وش صار في {name}؟",
        "احكي لي عن {name}",
        "متى حدث {name}؟",
        "وش قصة {name}؟",
        "وش تعرف عن {name}؟",
        "أبغى معلومات عن {name}",
        "اشرح لي {name}",
        "متى صار {name}؟",
        "ليش صار {name}؟",
        "ودي أعرف عن {name}"
    ],

    "HERITAGE": [
        "وش هو {name}؟",
        "وش معنى {name}؟",
        "احكي لي عن {name}",
        "وش تعرف عن {name}؟",
        "أبغى أعرف عن {name}",
        "اشرح لي {name}",
        "وش قصة {name}؟",
        "{name} وش يعتبر؟",
        "هل {name} من التراث؟",
        "وش نوع {name}؟",
        "عرفيني على {name}",
        "وش معنى {name} في التراث السعودي؟"
    ]
}


def generate_questions(entity_df, per_entity=6):
    out = []

    for _, row in entity_df.iterrows():
        label = row["label"]

        if label == "OTHER" or label not in TEMPLATES:
            continue

        variants = [row["name"]] + row["aliases"][:2]

        candidates = []

        for name in variants:
            for template in TEMPLATES[label]:
                candidates.append(
                    template.format(name=name)
                )

        # حذف التكرار
        candidates = list(set(candidates))

        random.shuffle(candidates)

        for text in candidates[:per_entity]:
            out.append({
                "text": text,
                "label": label,
                "entity_id": row["entity_id"]
            })

    return pd.DataFrame(out)


# توليد البيانات الأساسية
train_df = generate_questions(train_entities, 6)
val_df = generate_questions(val_entities, 4)
test_df = generate_questions(test_entities, 4)



# EVENT - أمثلة إضافية للتدريب


extra_event_train = [
    "متى حدث تأسيس الدولة السعودية؟",
    "وش صار وقت تأسيس الدولة السعودية؟",
    "احكي لي عن تأسيس الدولة السعودية",
    "أبغى أعرف عن حدث تأسيس الدولة السعودية",

    "متى صار توحيد المملكة؟",
    "وش قصة توحيد المملكة؟",
    "وش صار في توحيد المملكة؟",
    "احكي لي عن توحيد السعودية",

    "متى حدثت استعادة الرياض؟",
    "وش قصة استعادة الرياض؟",
    "وش صار في معركة استعادة الرياض؟",

    "احكي لي عن يوم التأسيس",
    "متى يوم التأسيس؟",
    "وش مناسبة اليوم الوطني؟",
    "ليش نحتفل باليوم الوطني؟"
]



# HERITAGE - أمثلة إضافية


extra_heritage_train = [
    "السدو وش يعتبر؟",
    "هل السدو من التراث؟",
    "وش نوع السدو؟",
    "السدو يعتبر مكان ولا تراث؟",
    "عرفيني على السدو",
    "وش معنى السدو في التراث السعودي؟",

    "الهريس وش يعتبر في التراث؟",
    "هل الهريس من التراث السعودي؟",
    "وش نوع الهريس؟",
    "وش معنى العنصر التراثي هذا؟"
]



# OTHER - خارج نطاق راوية


extra_other_train = [
    "كيف الجو اليوم؟",
    "كيف الجو بالرياض اليوم؟",
    "وش حالة الطقس بالرياض؟",
    "هل الجو حار اليوم؟",
    "كم درجة الحرارة؟",
    "كيف الطقس بكرة؟",
    "وش أخبار الطقس؟",

    "كم الساعة الحين؟",
    "كم الساعة الآن؟",
    "وش التاريخ اليوم؟",

    "وش أفضل جوال؟",
    "وش أفضل لابتوب؟",
    "وش أفضل مطعم؟",
    "اقترح لي مطعم",
    "اقترح لي وجبة",
    "اقترح لي فيلم",
    "وش أفضل مسلسل؟",

    "ساعدني أكتب إيميل",
    "ساعدني أكتب رسالة",
    "اكتب لي كود بايثون",

    "كيف أتعلم الإنجليزية؟",
    "كيف أتعلم البرمجة؟",
    "كيف أتعلم التصميم؟",

    "احسب لي 25 ضرب 18",
    "احسب لي 120 على 5",

    "ترجم لي هذه الجملة",
    "وش أخبار الرياضة؟",
    "كيف أسوي سيرة ذاتية؟",
    "كيف أحجز طيران؟",
    "وش سعر الدولار؟",
    "كيف أرتب وقتي؟",
    "وش أفضل تخصص جامعي؟",
    "كيف أنزل وزني؟",
    "وش أفضل تطبيق للملاحظات؟"
]
extra_other_train += [
    "هل بتمطر اليوم؟",
    "هل الجو غائم؟",
    "وش توقعات الطقس؟",

    "اكتب برنامج بسيط ببايثون",
    "ساعدني بكود بايثون",
    "اكتب لي سكربت بسيط",

    "رشح لي جوال جديد",
    "وش الجوال اللي تنصحني فيه؟",
    "رشح لي لابتوب",
    "رشح لي سماعة"
]


# إضافة أمثلة التدريب الإضافية
for text in extra_event_train:
    train_df.loc[len(train_df)] = [
        text,
        "EVENT",
        "extra_event"
    ]

for text in extra_heritage_train:
    train_df.loc[len(train_df)] = [
        text,
        "HERITAGE",
        "extra_heritage"
    ]

for text in extra_other_train:
    train_df.loc[len(train_df)] = [
        text,
        "OTHER",
        "extra_other"
    ]
    train_df = train_df.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)

print(train_df["label"].value_counts())



# Validation / Test لـ OTHER


other_val = [
    "اقترح لي أكلة للعشاء",
    "كم الوقت الآن؟",
    "كيف أطور لغتي الإنجليزية؟"
]

other_test = [
    "هل بتمطر اليوم؟",
    "اكتب برنامج بسيط ببايثون",
    "رشح لي جوال جديد"
]


for text in other_val:
    val_df.loc[len(val_df)] = [
        text,
        "OTHER",
        "other_val"
    ]

for text in other_test:
    test_df.loc[len(test_df)] = [
        text,
        "OTHER",
        "other_test"
    ]



# حذف أي تكرار


train_df = train_df.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)

val_df = val_df.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)

test_df = test_df.drop_duplicates(
    subset=["text", "label"]
).reset_index(drop=True)


print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTrain distribution:")
print(train_df["label"].value_counts())

display(
    train_df.sample(
        min(10, len(train_df)),
        random_state=SEED
    )
)

label
PLACE       342
HERITAGE    118
OTHER        44
EVENT        33
PERSON       18
Name: count, dtype: int64
Train: 555
Validation: 47
Test: 47

Train distribution:
label
PLACE       342
HERITAGE    118
OTHER        44
EVENT        33
PERSON       18
Name: count, dtype: int64


,text,label,entity_id
231,أين يقع قصر شبرا؟,PLACE,ar_7c9c8607b1fe
374,أين يقع مسجد اليمامة الأثري؟,PLACE,ar_660bc8fbf83e
55,وش معنى الصقارة: تراث إنساني حي؟,HERITAGE,ar_d6de3fd07ad5
381,أبغى معلومات عن قلعة عسفان,PLACE,ar_e9401a8caab9
70,احكي لي عن قصر الملك عبدالعزيز في الخرج,PLACE,ar_6d011b4ecfad
370,عرفيني على القط العسيري: الزخرفة الجدارية التق...,HERITAGE,ar_516e55a8aa77
376,احكي لي عن مسجد اليمامة الأثري,PLACE,ar_660bc8fbf83e
81,ودي أعرف وش السالفة ورا المربع,PLACE,ar_2fdd518e8659
513,وش حالة الطقس بالرياض؟,OTHER,extra_other
502,هل السدو من التراث؟,HERITAGE,extra_heritage


In [ ]:
print(train_df["label"].value_counts())

label
PLACE       342
HERITAGE    118
OTHER        34
EVENT        32
PERSON       18
Name: count, dtype: int64


In [ ]:
train_df.to_csv("rawiyah_train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv("rawiyah_validation.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("rawiyah_test.csv", index=False, encoding="utf-8-sig")
print("Dataset saved.")

Dataset saved.


 تجهيز AraBERT

In [ ]:
ArabertPreprocessor(model_name=MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prep(text):
    return arabert_prep.preprocess(str(text))

for df in [train_df, val_df, test_df]:
    df["text_processed"] = df["text"].apply(prep)
    df["labels"] = df["label"].map(label2id)

train_ds = Dataset.from_pandas(train_df[["text_processed","labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text_processed","labels"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["text_processed","labels"]], preserve_index=False)

def tokenize(batch):
    return tokenizer(batch["text_processed"], truncation=True, max_length=96)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

Map:   0%|          | 0/555 [00:00<?, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Map:   0%|          | 0/47 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


وحدات القياس

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": acc,
        "precision_macro": p,
        "recall_macro": r,
        "f1_macro": f1
    }

 التدريب

In [1]:
training_args = TrainingArguments(
    output_dir="./rawiyah_arabert_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics
)

trainer.train()

NameError: name 'TrainingArguments' is not defined

##  تقييم نهائي على Test Set

---



In [ ]:
metrics = trainer.evaluate(test_ds)
print(metrics)

pred = trainer.predict(test_ds)
pred_ids = np.argmax(pred.predictions, axis=-1)
true_ids = pred.label_ids

print(classification_report(
    true_ids,
    pred_ids,
    labels=list(range(len(LABELS))),
    target_names=LABELS,
    zero_division=0
))

results = test_df.reset_index(drop=True).copy()
results["predicted"] = [id2label[int(i)] for i in pred_ids]
results["correct"] = results["label"] == results["predicted"]
display(results[["text","label","predicted","correct"]])

Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.148719,0.196362,4,1.000000,1.000000,1.000000,1.000000


{'eval_loss': 0.19636180996894836, 'eval_accuracy': 1.0, 'eval_precision_macro': 1.0, 'eval_recall_macro': 1.0, 'eval_f1_macro': 1.0}


              precision    recall  f1-score   support

       PLACE       1.00      1.00      1.00        28
      PERSON       1.00      1.00      1.00         4
       EVENT       1.00      1.00      1.00         4
    HERITAGE       1.00      1.00      1.00         8
       OTHER       1.00      1.00      1.00         3

    accuracy                           1.00        47
   macro avg       1.00      1.00      1.00        47
weighted avg       1.00      1.00      1.00        47



,text,label,predicted,correct
0,وش أهمية جدة التاريخية، بوابة مكة؟,PLACE,PLACE,True
1,ودي أعرف وش السالفة ورا جدة التاريخية، بوابة مكة,PLACE,PLACE,True
2,كلمني عن جدة التاريخية، بوابة مكة,PLACE,PLACE,True
3,وش تعرف عن جدة التاريخية، بوابة مكة؟,PLACE,PLACE,True
4,متى حدث توحيد المملكة العربية السعودية؟,EVENT,EVENT,True
5,وش تعرف عن تأسيس المملكة العربية السعودية؟,EVENT,EVENT,True
6,متى حدث تأسيس المملكة العربية السعودية؟,EVENT,EVENT,True
7,أبغى معلومات عن تأسيس المملكة العربية السعودية,EVENT,EVENT,True
8,وش تعرف عن قصر الزاهر؟,PLACE,PLACE,True
9,وش قصة قصر الزاهر؟,PLACE,PLACE,True




---



In [ ]:
def predict_question(question):
    text = arabert_prep.preprocess(question)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=96)
    device = trainer.model.device
    inputs = {k:v.to(device) for k,v in inputs.items()}

    trainer.model.eval()
    with torch.no_grad():
        out = trainer.model(**inputs)
        probs = torch.softmax(out.logits, dim=-1)[0]

    pred_id = int(torch.argmax(probs).item())
    return {
        "question": question,
        "label": id2label[pred_id],
        "confidence": round(float(probs[pred_id].item()), 4)
    }

for q in [
    "وش قصة قصر المصمك؟",
    "من هو الملك عبدالعزيز؟",
    "متى صار توحيد المملكة؟",
    "وش معنى السدو؟",
    "وش أفضل جوال؟"
]:
    print(predict_question(q))

{'question': 'وش قصة قصر المصمك؟', 'label': 'PLACE', 'confidence': 0.965}
{'question': 'من هو الملك عبدالعزيز؟', 'label': 'PERSON', 'confidence': 0.3683}
{'question': 'متى صار توحيد المملكة؟', 'label': 'EVENT', 'confidence': 0.4675}
{'question': 'وش معنى السدو؟', 'label': 'HERITAGE', 'confidence': 0.9443}
{'question': 'وش أفضل جوال؟', 'label': 'OTHER', 'confidence': 0.5349}


In [ ]:
final_real_questions = [
    "وش السالفة ورا المصمك؟",
    "عرفيني بالإمام محمد بن سعود",
    "متى اتوحدت المملكة؟",
    "السدو يعتبر تراث ولا مكان؟",
    "وش اللي صار في استعادة الرياض؟",
    "وين حي الطريف؟",
    "من هو الإمام تركي بن عبدالله؟",
    "ليش يحتفلون بيوم التأسيس؟",
    "الهريس وش علاقته بالتراث؟",
    "هل الجو بيبرد الليلة؟",
    "سو لي برنامج يحسب المتوسط",
    "وش الجوال اللي تنصحني فيه؟"
]

for q in final_real_questions:
    print(predict_question(q))

{'question': 'وش السالفة ورا المصمك؟', 'label': 'PLACE', 'confidence': 0.9748}
{'question': 'عرفيني بالإمام محمد بن سعود', 'label': 'PERSON', 'confidence': 0.5231}
{'question': 'متى اتوحدت المملكة؟', 'label': 'EVENT', 'confidence': 0.4631}
{'question': 'السدو يعتبر تراث ولا مكان؟', 'label': 'HERITAGE', 'confidence': 0.96}
{'question': 'وش اللي صار في استعادة الرياض؟', 'label': 'EVENT', 'confidence': 0.4857}
{'question': 'وين حي الطريف؟', 'label': 'PLACE', 'confidence': 0.9641}
{'question': 'من هو الإمام تركي بن عبدالله؟', 'label': 'PERSON', 'confidence': 0.4556}
{'question': 'ليش يحتفلون بيوم التأسيس؟', 'label': 'EVENT', 'confidence': 0.4133}
{'question': 'الهريس وش علاقته بالتراث؟', 'label': 'HERITAGE', 'confidence': 0.9488}
{'question': 'هل الجو بيبرد الليلة؟', 'label': 'OTHER', 'confidence': 0.4469}
{'question': 'سو لي برنامج يحسب المتوسط', 'label': 'OTHER', 'confidence': 0.5498}
{'question': 'وش الجوال اللي تنصحني فيه؟', 'label': 'OTHER', 'confidence': 0.5448}


In [ ]:
real_questions = [
    "ودي أعرف وش السالفة ورا قصر المصمك",
    "مين الإمام محمد بن سعود؟",
    "متى توحدت السعودية بشكل رسمي؟",
    "السدو وش يعتبر بالضبط؟",
    "وش قصة استرجاع الرياض؟",
    "كلمني عن الطريف",
    "من هو تركي بن عبدالله؟",
    "ليش نحتفل بيوم التأسيس؟",
    "وش معنى الهريس في التراث؟",
    "كيف الجو بالرياض اليوم؟",
    "ساعدني أكتب إيميل",
    "وين تقع الدرعية؟",
]

for q in real_questions:
    print(predict_question(q))

{'question': 'ودي أعرف وش السالفة ورا قصر المصمك', 'label': 'PLACE', 'confidence': 0.9788}
{'question': 'مين الإمام محمد بن سعود؟', 'label': 'PERSON', 'confidence': 0.6501}
{'question': 'متى توحدت السعودية بشكل رسمي؟', 'label': 'EVENT', 'confidence': 0.4096}
{'question': 'السدو وش يعتبر بالضبط؟', 'label': 'PLACE', 'confidence': 0.6793}
{'question': 'وش قصة استرجاع الرياض؟', 'label': 'EVENT', 'confidence': 0.4031}
{'question': 'كلمني عن الطريف', 'label': 'PLACE', 'confidence': 0.9495}
{'question': 'من هو تركي بن عبدالله؟', 'label': 'PERSON', 'confidence': 0.5124}
{'question': 'ليش نحتفل بيوم التأسيس؟', 'label': 'EVENT', 'confidence': 0.5171}
{'question': 'وش معنى الهريس في التراث؟', 'label': 'HERITAGE', 'confidence': 0.8494}
{'question': 'كيف الجو بالرياض اليوم؟', 'label': 'EVENT', 'confidence': 0.2807}
{'question': 'ساعدني أكتب إيميل', 'label': 'OTHER', 'confidence': 0.5231}
{'question': 'وين تقع الدرعية؟', 'label': 'PLACE', 'confidence': 0.9806}


##  حفظ المودل النهائي

In [ ]:
FINAL_DIR = "rawiyah-arabert"

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

metadata = {
    "base_model": MODEL_NAME,
    "labels": LABELS,
    "label2id": label2id,
    "id2label": {str(k):v for k,v in id2label.items()},
    "seed": SEED,
    "task": "Rawiyah Arabic question type classification"
}
with open(os.path.join(FINAL_DIR, "rawiyah_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("Saved:", FINAL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: rawiyah-arabert


##  ضغط المودل وتنزيله




In [ ]:
from google.colab import files

zip_path = shutil.make_archive(
    "rawiyah-arabert",
    "zip",
    root_dir=FINAL_DIR
)

print(zip_path)
files.download(zip_path)

/content/rawiyah-arabert.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>